In [ ]:

# The code below converts a CSV file into a custom XML file that is usable for DCaR ASN Matching submissions
# ***IMPORTANT***: Save and close the relevant CSV and XML files before running this script

# Import modules
import csv
import xml.etree.ElementTree as ET
import xml.dom.minidom

# Place the CSV file in the same folder as this script
# The CSV data must have the following columns: 
#   - "Provider"
#   - "StudentID"
#   - "ASN"
#   - "FirstName"
#   - "MiddleName"
#   - "LastName"
#   - "Birthdate"
#   - "Gender"
#   - "FormerLastName"
#   - "AKAGivenNames"

# Open the csv file in read mode
with open("data_extract.csv", mode="r") as file:
    reader = csv.DictReader(file)

    # Create an empty list called "data_dictionary" to hold enrolment data
    data_list = []

    # For each row of data stored in the "reader" object, store the data values in the data_list list"
    for row in reader:
        data_list.append({
            "attributes": {
                "Provider": row["Provider"],
                "StudentID": row["StudentID"],
                "ASN": row["ASN"],
                "FirstName": row["FirstName"],
                "MiddleName": row["MiddleName"],
                "LastName": row["LastName"],
                "Birthdate": row["Birthdate"],
                "Gender": row["Gender"],
                "FormerLastName": row["FormerLastName"],
                "AKAGivenNames": row["AKAGivenNames"]
            }
        })

# Sort data_list by StudentID in ascending lexicographical order, store the sorted StudentIDs in a list called "sorted_student_ids"
sorted_student_ids = sorted(data_list, key=lambda x: x["attributes"]["StudentID"])

# Create the root element <ASN xmlns="http://psdata.eae.alberta.ca/asn/1"></ASN>
asn_element = ET.Element("ASN", xmlns="http://psdata.eae.alberta.ca/asn/1")

# Loop through each student_id in "sorted_student_ids" and retrieve all data in "data_list"
# This will retrieve data from "data_list" in ascending order by StudentIDs
for student_data in sorted_student_ids:

    # Create the "Student" element: <Student ... />
    student_element = ET.SubElement(asn_element, "Student", **student_data["attributes"])

# The "student_element" variable now stores a XML tree object with all student data
# ET.tostring() converts the XML tree object (lers_element) into raw XML bytes
# The encoding="utf-8" argument outputs a bytes object
raw_xml_string = ET.tostring(asn_element, encoding="utf-8")

# xml.dom.minidom is a DOM (Document Object Model) parser
# parseString() takes "raw_xml_string" from the previous step and parses it into a DOM object — a structured, in-memory representation that allows easy formatting and traversal.
xml_parsed_DOM_object = xml.dom.minidom.parseString(raw_xml_string)

# toprettyxml() converts the DOM object ("xml_parsed_DOM_object") back into a readable XML string with indentation and newlines
# indent="  " adds 2 spaces in front of every Parent element and every Child element
# newl="\n" writes every element on a new line
# xml.dom.minidom.toprettyxml() adds the XML declaration of <?xml version="1.0"?> before the root element by default, encoding="utf-8" turns this XML declaration into <?xml version="1.0" encoding="utf-8"?>
# decode("utf-8") converts bytes into Python string
xml_readable_string = xml_parsed_DOM_object.toprettyxml(indent="  ", newl="\n", encoding="utf-8").decode("utf-8")

# Child (Enrolment) elements require a space before its closing syntax '/>' 
# Since only Child elements end in '"/>", replacing all '"/>" with '" />' will only add a space before the closing syntax of Child elements
xml_readable_string = xml_readable_string.replace('"/>', '" />')

# Remove all spaces at the end of the XML string, meaning remove all spaces after </LERS>
xml_readable_string = xml_readable_string.rstrip()

# Open a XML file named "LERS_CONVERTED_DATA.xml" in binary write mode ("wb")
# Write the UTF-8 Byte Order Mark (BOM) at the start of the file
# Write the XML string "xml_readable_string" as UTF-8 bytes
# Close the XML file
with open("ASN_CONVERTED_DATA.xml", "wb") as xml_file:
    xml_file.write(b'\xef\xbb\xbf')  # Write Byte BOM for UTF-8 encoding
    xml_file.write(xml_readable_string.encode('utf-8'))